In [0]:
-- only the latest weekly copy. specs barely change so no history kept here
create or replace table cloud_pricing.silver.azure_vm_specs as
with raw as (
select regexp_extract(_metadata.file_path, 'run_date=([0-9-]+)', 1) run_date,
regexp_extract(_metadata.file_path, 'run_date=[0-9-]+/([^/]+)/', 1) region,
explode(value) s
from read_files('/Volumes/cloud_pricing/raw/landing/azure_vm_skus', format => 'json', recursiveFileLookup => 'true')
)
select region, s.name sku_name, s.family family,
cast(map_from_entries(transform(s.capabilities, c -> struct(c.name, c.value)))['vCPUs'] as int) vcpus,
cast(map_from_entries(transform(s.capabilities, c -> struct(c.name, c.value)))['MemoryGB'] as decimal(10,2)) memory_gb,
lower(map_from_entries(transform(s.capabilities, c -> struct(c.name, c.value)))['CpuArchitectureType']) cpu_arch,
-- some sizes exist in the region but this subscription can't create them
coalesce(size(s.restrictions), 0) > 0 restricted_for_subscription,
to_date(run_date) specs_date
from raw
where s.resourceType = 'virtualMachines' and run_date = (select max(run_date) from raw)

In [0]:
select region, assert_true(count(*) = count(distinct sku_name), 'same vm size twice in a region')
from cloud_pricing.silver.azure_vm_specs group by region

In [0]:
-- how many priced vms in silver actually got specs
select p.region, count(distinct p.arm_sku_name) priced_skus, count(distinct s.sku_name) with_specs
from cloud_pricing.silver.vm_prices p
left join cloud_pricing.silver.azure_vm_specs s on s.region = p.region and s.sku_name = p.arm_sku_name
where p.observed_at = (select max(observed_at) from cloud_pricing.silver.vm_prices)
group by p.region